# 10. Feature Engineering (피처 엔지니어링)

## 목적
전처리된 데이터에서 ML 학습용 파생 피처 생성

## 입력
- `preprocessed.csv`: 전처리된 데이터 (결측 처리 완료)

## 출력
- `features_final.csv`: 최종 피처셋 (35개 피처)

## 최종 피처 구성 (35개)
| 구분 | 피처 | 개수 |
|------|------|------|
| Vitals | hr, rr, spo2, sbp, dbp, mbp, temp | 7 |
| Vitals 통계 | hr_max, rr_max, spo2_min, sbp_min | 4 |
| Labs | creatinine, wbc, platelets, potassium, sodium, lactate | 6 |
| GCS | gcs_eye, gcs_verbal, gcs_motor, gcs_total | 4 |
| Urine | urine_ml_6h, urine_ml_kg_hr_avg, oliguria_flag | 3 |
| 파생 | shock_index, anchor_age, news_score, mews_score | 4 |
| Flags | lactate_missing, abga_checked, is_readmission | 3 |
| Delta | hr_delta, sbp_delta, spo2_delta, lactate_delta, gcs_total_delta | 5 |

## 피처 엔지니어링 범위
- [x] 파생 피처 (Shock Index)
- [x] 중증도 점수 (NEWS, MEWS) - 벡터화
- [x] Delta 피처 (5개)
- [x] 인구통계학적 피처 (anchor_age)

In [ ]:
import pandas as pd
import numpy as np
import os

INPUT_DIR = '../data/processed'
OUTPUT_DIR = '../data/processed'

print("=== 10. Feature Engineering 시작 ===")

## Step 1: 데이터 로드

In [ ]:
print("Step 1: 데이터 로드")

df = pd.read_csv(os.path.join(INPUT_DIR, 'preprocessed.csv'))
df = df.sort_values(['stay_id', 'observation_hour']).reset_index(drop=True)

print(f"✓ 데이터 로드 완료: {len(df):,} rows")
print(f"\n고유 환자: {df['stay_id'].nunique():,}명")
print(f"\n고유 stay: {df['stay_id'].nunique():,}개")
print(f"\n컬럼 수: {len(df.columns)}개")

In [ ]:
# 피처 그룹 정의 (전처리에서 넘어온 것)
vital_cols = ['hr', 'rr', 'spo2', 'sbp', 'dbp', 'mbp', 'temp']
vital_stat_cols = ['hr_max', 'rr_max', 'spo2_min', 'sbp_min']  # 08에서 생성
lab_cols = ['creatinine', 'wbc', 'platelets', 'potassium', 'sodium', 'lactate']
gcs_cols = ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total']
urine_cols = ['urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag']

# 전처리에서 생성된 플래그
flag_cols = ['lactate_missing', 'abga_checked', 'is_readmission']

# 현재 존재하는 피처 확인
print("\n=== 입력 피처 확인 ===")
for name, cols in [('Vitals', vital_cols), ('Vital Stats', vital_stat_cols), 
                   ('Labs', lab_cols), ('GCS', gcs_cols), ('Urine', urine_cols),
                   ('Flags', flag_cols)]:
    existing = [c for c in cols if c in df.columns]
    missing = [c for c in cols if c not in df.columns]
    print(f"  {name}: {len(existing)}/{len(cols)} 존재")
    if missing:
        print(f"    ⚠️ 없음: {missing}")

## Step 2: Shock Index (파생 피처)

**Shock Index = HR / SBP**
- 정상: 0.5 ~ 0.7
- \>1.0: 쇼크 의심

SHAP 분석에서 상위 피처로 확인됨

In [ ]:
print("Step 2: Shock Index 생성")

# Shock Index = HR / SBP
df['shock_index'] = df['hr'] / df['sbp'].replace(0, np.nan)
df['shock_index'] = df['shock_index'].fillna(df['shock_index'].median())
df['shock_index'] = df['shock_index'].clip(0, 3)  # 이상치 클리핑

print(f"✓ shock_index: mean={df['shock_index'].mean():.3f}, median={df['shock_index'].median():.3f}")
print(f"  range: {df['shock_index'].min():.3f} ~ {df['shock_index'].max():.3f}")

## Step 3: 중증도 점수 (MEWS, NEWS) - 벡터화

### MEWS (Modified Early Warning Score)
- HR, SBP, RR, Temp 기반
- 범위: 0~14

### NEWS (National Early Warning Score)
- SpO2, SBP, HR, RR, Temp, GCS 기반
- 범위: 0~20

**벡터화**: `apply(axis=1)` 대신 `np.where` 사용 → 5~15분 → 2~5초

In [ ]:
print("Step 3: 중증도 점수 계산 (벡터화)")

# --- MEWS 벡터화 ---
def calc_mews_vectorized(df):
    """
    Modified Early Warning Score (벡터화)
    - HR: 40미만/130초과(3), 50미만/110초과(2), 60미만/100초과(1)
    - SBP: 70미만(3), 80미만(2), 100미만(1)
    - RR: 9미만/30초과(3), 25초과(2), 12미만/20초과(1)
    - Temp: 35미만/39초과(2), 36미만/38초과(1)
    """
    score = pd.Series(0, index=df.index)
    
    # HR
    hr = df['hr']
    score += np.where((hr < 40) | (hr > 130), 3,
             np.where((hr < 50) | (hr > 110), 2,
             np.where((hr < 60) | (hr > 100), 1, 0)))
    
    # SBP
    sbp = df['sbp']
    score += np.where(sbp < 70, 3,
             np.where(sbp < 80, 2,
             np.where(sbp < 100, 1, 0)))
    
    # RR
    rr = df['rr']
    score += np.where((rr < 9) | (rr > 30), 3,
             np.where(rr > 25, 2,
             np.where((rr < 12) | (rr > 20), 1, 0)))
    
    # Temp
    temp = df['temp']
    score += np.where((temp < 35) | (temp > 39), 2,
             np.where((temp < 36) | (temp > 38), 1, 0))
    
    return score

df['mews_score'] = calc_mews_vectorized(df)
print(f"✓ mews_score: mean={df['mews_score'].mean():.2f}, max={df['mews_score'].max()}")

In [ ]:
# --- NEWS 벡터화 ---
def calc_news_vectorized(df):
    """
    National Early Warning Score (벡터화)
    - RR: 8이하/25이상(3), 21-24(2), 9-11(1)
    - SpO2: 91이하(3), 92-93(2), 94-95(1)
    - Temp: 35이하/39.1이상(3), 35.1-36/38.1-39(1)
    - SBP: 90이하/220이상(3), 91-100(2), 101-110(1)
    - HR: 40이하/131이상(3), 111-130(2), 41-50/91-110(1)
    - Consciousness: GCS<15 → +3
    """
    score = pd.Series(0, index=df.index)
    
    # RR
    rr = df['rr']
    score += np.where((rr <= 8) | (rr >= 25), 3,
             np.where((rr >= 21) & (rr <= 24), 2,
             np.where((rr >= 9) & (rr <= 11), 1, 0)))
    
    # SpO2
    spo2 = df['spo2']
    score += np.where(spo2 <= 91, 3,
             np.where(spo2 <= 93, 2,
             np.where(spo2 <= 95, 1, 0)))
    
    # Temp
    temp = df['temp']
    score += np.where((temp <= 35) | (temp >= 39.1), 3,
             np.where(((temp > 35) & (temp <= 36)) | ((temp >= 38.1) & (temp < 39.1)), 1, 0))
    
    # SBP
    sbp = df['sbp']
    score += np.where((sbp <= 90) | (sbp >= 220), 3,
             np.where((sbp >= 91) & (sbp <= 100), 2,
             np.where((sbp >= 101) & (sbp <= 110), 1, 0)))
    
    # HR
    hr = df['hr']
    score += np.where((hr <= 40) | (hr >= 131), 3,
             np.where((hr >= 111) & (hr <= 130), 2,
             np.where(((hr >= 41) & (hr <= 50)) | ((hr >= 91) & (hr <= 110)), 1, 0)))
    
    # Consciousness (GCS 기반)
    gcs = df['gcs_total']
    score += np.where(gcs < 15, 3, 0)
    
    return score

df['news_score'] = calc_news_vectorized(df)
print(f"✓ news_score: mean={df['news_score'].mean():.2f}, max={df['news_score'].max()}")

## Step 4: Delta 피처 (변화량)

이전 시점 대비 변화량 - SHAP 분석에서 일부 유효성 확인

- **hr_delta**: 심박수 변화
- **sbp_delta**: 수축기혈압 변화
- **spo2_delta**: 산소포화도 변화
- **lactate_delta**: 젖산 변화
- **gcs_total_delta**: GCS 변화

In [ ]:
print("Step 4: Delta 피처 생성")

delta_features = ['hr', 'sbp', 'spo2', 'lactate', 'gcs_total']

for col in delta_features:
    if col not in df.columns:
        print(f"  ⚠️ {col} 없음, 스킵")
        continue
    
    delta_col = f'{col}_delta'
    
    # 환자별 이전 시점 대비 변화량
    df[delta_col] = df.groupby('stay_id')[col].diff()
    
    # 첫 번째 윈도우는 결측 → 0으로 채움 (변화 없음)
    df[delta_col] = df[delta_col].fillna(0)
    
    print(f"  ✓ {delta_col}: mean={df[delta_col].mean():.3f}, std={df[delta_col].std():.3f}")

print(f"\n✓ Delta 피처 {len(delta_features)}개 생성 완료")

## Step 5: anchor_age 확인

나이는 08_sliding_window_merge에서 cohort_base와 병합됨

In [ ]:
print("Step 5: anchor_age 확인")

if 'anchor_age' in df.columns:
    print(f"✓ anchor_age 존재")
    print(f"  mean: {df['anchor_age'].mean():.1f}")
    print(f"  range: {df['anchor_age'].min()} ~ {df['anchor_age'].max()}")
else:
    print("⚠️ anchor_age 없음 - 08에서 cohort_base 병합 확인 필요")

## Step 6: 최종 피처 정리 및 검증

In [ ]:
print("Step 6: 최종 피처 정리")

# 최종 35개 피처 정의
final_features = {
    'vitals': ['hr', 'rr', 'spo2', 'sbp', 'dbp', 'mbp', 'temp'],
    'vitals_stat': ['hr_max', 'rr_max', 'spo2_min', 'sbp_min'],
    'labs': ['creatinine', 'wbc', 'platelets', 'potassium', 'sodium', 'lactate'],
    'gcs': ['gcs_eye', 'gcs_verbal', 'gcs_motor', 'gcs_total'],
    'urine': ['urine_ml_6h', 'urine_ml_kg_hr_avg', 'oliguria_flag'],
    'derived': ['shock_index', 'anchor_age', 'news_score', 'mews_score'],
    'flags': ['lactate_missing', 'abga_checked', 'is_readmission'],
    'delta': ['hr_delta', 'sbp_delta', 'spo2_delta', 'lactate_delta', 'gcs_total_delta']
}

# 전체 피처 리스트
all_features = []
for group, cols in final_features.items():
    all_features.extend(cols)

print(f"\n=== 최종 피처 구성 ({len(all_features)}개) ===")
for group, cols in final_features.items():
    existing = [c for c in cols if c in df.columns]
    missing_cols = [c for c in cols if c not in df.columns]
    status = "✓" if len(missing_cols) == 0 else "⚠️"
    print(f"  {status} [{group}] {len(existing)}/{len(cols)}개")
    if missing_cols:
        print(f"      없음: {missing_cols}")

In [ ]:
# 결측 확인
print("\n=== 결측 확인 ===")
existing_features = [c for c in all_features if c in df.columns]
total_missing = df[existing_features].isna().sum().sum()

if total_missing == 0:
    print(f"✓ {len(existing_features)}개 피처 모두 결측 없음")
else:
    print(f"⚠️ 총 결측: {total_missing:,}건")
    missing_detail = df[existing_features].isna().sum()
    print(missing_detail[missing_detail > 0])

In [ ]:
# 피처 기술 통계
print("\n=== 피처 기술 통계 ===")
print(df[existing_features].describe().round(2))

## Step 7: 최종 DataFrame 구성

In [ ]:
print("Step 7: 최종 DataFrame 구성")

# ID 컬럼
id_cols = ['stay_id', 'subject_id', 'hadm_id', 'observation_hour', 'observation_start', 'observation_end']

# 레이블 컬럼
label_cols = [col for col in df.columns if 'next_' in col]

# 최종 컬럼 순서
final_cols = (
    [c for c in id_cols if c in df.columns] +
    [c for c in all_features if c in df.columns] +
    label_cols
)

df_final = df[final_cols].copy()

print(f"\n=== 최종 DataFrame ===")
print(f"  행 수: {len(df_final):,}")
print(f"  컬럼 수: {len(df_final.columns)}")
print(f"    - ID: {len([c for c in id_cols if c in df_final.columns])}개")
print(f"    - 피처: {len([c for c in all_features if c in df_final.columns])}개")
print(f"    - 레이블: {len(label_cols)}개")

## Step 8: 저장

In [ ]:
print("Step 8: 저장")

output_path = os.path.join(OUTPUT_DIR, 'features_final.csv')
df_final.to_csv(output_path, index=False)

file_size = os.path.getsize(output_path) / (1024 * 1024)

print(f"\n✓ 저장 완료: features_final.csv")
print(f"  - 파일 크기: {file_size:.2f} MB")
print(f"  - 행 수: {len(df_final):,}개")
print(f"  - 컬럼 수: {len(df_final.columns)}개")
print(f"  - 경로: {output_path}")

In [ ]:
# 피처 목록 출력
print(f"\n=== 최종 피처 목록 ({len(existing_features)}개) ===")
for group, cols in final_features.items():
    print(f"\n[{group.upper()}]")
    for col in cols:
        if col in df_final.columns:
            print(f"  ✓ {col}")
        else:
            print(f"  ✗ {col} (없음)")

In [ ]:
# 레이블 분포
print(f"\n=== 레이블 분포 ===")
for col in sorted(label_cols):
    pos_rate = df_final[col].mean() * 100
    pos_count = df_final[col].sum()
    print(f"  {col}: {pos_count:,} ({pos_rate:.2f}%)")

In [ ]:
print("\n" + "="*50)
print("=== 10. Feature Engineering 완료 ===")
print("="*50)
print(f"\n최종 피처: {len(existing_features)}개")
print(f"샘플 수: {len(df_final):,}개")
print(f"환자 수: {df_final['stay_id'].nunique():,}명")

---
## +a: 검증

In [ ]:
# 결측 최종 확인
print("=== 결측 최종 확인 ===")
feature_cols_only = [c for c in all_features if c in df_final.columns]
missing = df_final[feature_cols_only].isna().sum()
if missing.sum() == 0:
    print("✓ 모든 피처 결측 없음")
else:
    print("결측 있는 피처:")
    print(missing[missing > 0])

In [ ]:
# 고상관 피처 확인 (다중공선성)
print("\n=== 고상관 피처 쌍 (>0.95) ===")
feature_cols_only = [c for c in all_features if c in df_final.columns]
corr = df_final[feature_cols_only].corr()

high_corr = []
for i in range(len(corr.columns)):
    for j in range(i+1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.95:
            high_corr.append((corr.columns[i], corr.columns[j], corr.iloc[i, j]))

if high_corr:
    for pair in high_corr[:10]:
        print(f"  {pair[0]} vs {pair[1]}: {pair[2]:.3f}")
else:
    print("  없음")

In [ ]:
# Flags 분포 확인
print("\n=== Flags 분포 ===")
for col in ['lactate_missing', 'abga_checked']:
    if col in df_final.columns:
        ones = (df_final[col] == 1).sum()
        zeros = (df_final[col] == 0).sum()
        print(f"  {col}: 1={ones:,} ({ones/len(df_final)*100:.1f}%), 0={zeros:,} ({zeros/len(df_final)*100:.1f}%)")

In [ ]:
# 샘플 데이터 확인
print("\n=== 샘플 데이터 ===")
df_final.head()